# 01 — Smoke Test(端到端煙測)

**用途**:用 2 段 URFD 短片(1 跌倒 + 1 日常)驗證整條 pipeline:
下載 → PNG 序列重組影片 → YOLO26-pose + ByteTrack 抽關鍵點 → 規則引擎 → 標註影片 + events.json。

**執行方式**:`Runtime → Run all`(建議 GPU runtime;T4 即可,全程約 5 分鐘)。
跑完後把最後一格的 JSON 摘要貼回對話。

**預期觀察**(文獻起點閾值,尚未在 tune split 校準):
- `fall-01` 應偵測到 **1 個跌倒事件**,標註影片中框色 UPRIGHT(綠)→ FALLING(橘)→ FALLEN/ALARM(紅)+ 頂部紅色警示;
- `adl-01` 應為 **0 個事件**,全程綠框。

In [ ]:
!nvidia-smi || echo '無 GPU:CPU 也能跑,只是比較慢'

In [ ]:
import os
if os.path.basename(os.getcwd()) != 'fall-detection-pose':
    if not os.path.exists('fall-detection-pose'):
        !git clone -q https://github.com/tun0000/fall-detection-pose.git
    %cd fall-detection-pose
!pip -q install -e ".[infer]" pytest
import fall_detection
print('fall_detection', fall_detection.__version__)

In [ ]:
# 規則引擎單元測試(純 CPU、合成軌跡,~10 秒):必須全綠
!python -m pytest -q

In [ ]:
# 下載 2 段 URFD 序列(官方站,CC BY-NC-SA 4.0;僅本 VM 暫存,不進 git)
from fall_detection.io import urfd

DATA_DIR = 'data/urfd_smoke'
SEQS = ['fall-01', 'adl-01']
urfd.download_annotations(DATA_DIR)
urfd.download_sequences(DATA_DIR, SEQS)
videos = urfd.build_videos(DATA_DIR, SEQS)
videos

In [ ]:
# extract(GPU)→ 規則引擎(CPU)→ 標註影片 + events.json
import torch
from fall_detection.config import load_config
from fall_detection.events.schema import write_events_json
from fall_detection.inference.extract import extract_video
from fall_detection.io.cache import read_cache
from fall_detection.rules import run_engine
from fall_detection.viz.annotate import annotate_video

cfg = load_config('config.yaml')
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('model =', cfg.model.name, '| device =', device)

summary = {}
for seq, video in videos.items():
    cache_path = f'outputs/smoke/{seq}.parquet'
    extract_video(video, cache_path, cfg, device=device)
    df, meta = read_cache(cache_path)
    events, debug = run_engine(df, meta.fps, cfg, collect_debug=True)
    annotate_video(video, df, meta.fps, cfg, events, debug,
                   f'outputs/smoke/{seq}_annotated.mp4')
    write_events_json(f'outputs/smoke/{seq}.events.json', events,
                      source=str(video), fps=meta.fps)
    summary[seq] = [e.to_dict() for e in events]
    print(f'{seq}: {len(events)} 個事件, cache rows={len(df)}')

In [ ]:
# 與官方標註對照(GT 慣例:[第一個 label=0 幀, 連續 label=1 區段末幀])
from fall_detection.eval.ground_truth import load_gt_events

gt = load_gt_events(f'{DATA_DIR}/urfall-cam0-falls.csv')
print('fall-01 GT 事件(幀):', gt.get('fall-01'))
print('fall-01 預測事件(幀):',
      [(e['start_frame'], e['end_frame']) for e in summary['fall-01']])

In [ ]:
# 內嵌播放標註影片(H.264,瀏覽器可直接播)
from base64 import b64encode
from IPython.display import HTML, display

def show(path, width=480):
    data = b64encode(open(path, 'rb').read()).decode()
    display(HTML(
        f'<p><b>{path}</b></p>'
        f'<video width={width} controls src="data:video/mp4;base64,{data}"></video>'))

show('outputs/smoke/fall-01_annotated.mp4')
show('outputs/smoke/adl-01_annotated.mp4')

In [ ]:
# ===== 煙測摘要:請把這格輸出貼回對話 =====
import json, sys
import ultralytics

report = {
    'python': sys.version.split()[0],
    'ultralytics': ultralytics.__version__,
    'torch': torch.__version__,
    'device': device,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gt_fall01_frames': gt.get('fall-01'),
    'events': {
        seq: [
            {k: e[k] for k in ('track_ids', 'start_frame', 'end_frame',
                               'start_time_s', 'end_time_s', 'rules_fired')}
            for e in evs
        ]
        for seq, evs in summary.items()
    },
}
print('=== SMOKE TEST 摘要(請貼回對話) ===')
print(json.dumps(report, ensure_ascii=False, indent=2))